# Problem 5: Parallelising the Bible Code Search

---

## Background

In Problem 4 your ELS program searches 999 strides across three texts sequentially — one stride at a time. This problem is **embarrassingly parallel**: each stride is completely independent of every other, meaning all 999 strides across all three texts can be distributed across CPU cores with no coordination required.

Python's `multiprocessing` module makes this straightforward. Rather than looping through strides one by one, you define a worker function that handles a single `(stride, text)` task, build a list of all tasks, and hand them to a pool of workers that run simultaneously.

---

## Task

Rewrite your Problem 4 program to use `multiprocessing`. The starter structure below shows an example approach — your job is to fill in the missing pieces, in particular the `find_matches` function and the result-writing logic. Feel free to design your own approach, rename, or rafactor this example.

```python
from multiprocessing import Pool

# worker function — processes one stride against one text
def process_stride(args):
    stride, text_name, text, word_list = args
    extracted = text[::stride]
    matches = find_matches(extracted, stride, word_list)
    return text_name, matches

# build a flat list of every (stride, text) task
tasks = []
for text_name, text in texts.items():
    for stride in range(2, 1001):
        tasks.append((stride, text_name, text, word_list))

# open a pool of workers — one per available CPU core
# all tasks are sent at once; workers run simultaneously
with Pool() as pool:
    all_results = pool.map(process_stride, tasks)
```

> **Note:** `multiprocessing` spawns separate processes rather than threads, so large objects like full text strings are copied to each worker. For very large texts this memory overhead can offset the speed gain — consider passing file paths and loading the text inside each worker instead.

**Speedup** is a metric used to assess the relative performance improvement gained by executing a task one way versus another. In our case we are comparing serial execution (a single process) against a parallelised version. Speedup $S$ is defined as:

$S = \frac{T_{\text{old}}}{T_{\text{new}}}$

where $T_{\text{old}}$ is the wall time of the serial run and $T_{\text{new}}$ is the wall time of the parallel run. A speedup of $2.0$ means the parallel version ran twice as fast; a speedup of $1.0$ means no improvement.

Use a timing function to measure wall time with and without multiprocessing for each text independently. Record your results in the table below and calculate the speedup for each.

| Text | Sequential ($T_{\text{old}}$) | Parallel ($T_{\text{new}}$) | Speedup ($S$) |
|---|---|---|---|
| Bible | | | |
| War and Peace | | | |
| Moby Dick | | | |

On a quad-core machine expect $S \approx 3$–$4$; on an 8-core machine, closer to $6$–$7$. If your speedup is lower than expected, what might explain it? Do all three texts benefit equally, and if not, why?


| Text | Sequential ($T_{\text{old}}$) | Parallel ($T_{\text{new}}$) | Speedup ($S$) |
|---|---|---|---|
| Bible | 249.29| 84.45 | 2.95 |
| War and Peace | 196.04 | 67.27 |2.91 |
| Moby Dick | 76.46 | 28.17 | 2.71|

On my 8-core machine, multiprocessing achieved speedups of 2.95× for the Bible, 2.91× for War and Peace, and 2.71× for Moby Dick, which are below the expected 6×–7×. This could be explained by multiprocessing overhead, including worker initialization, interprocess communication, and transferring results between processes. Additionally, smaller strides require more computation than larger strides, leading to uneven workload distribution among workers. 

All three texts benefited from multiprocessing, but not equally. The Bible achieved the highest speedup (2.95×), while Moby Dick achieved the lowest (2.71×). This is likely because longer-running workloads can better amortize multiprocessing overhead, whereas shorter workloads spend a larger proportion of their execution time on process management and communication.